# 3. Aperiodic proteins: apo and holo-derived calmodulin

**Question:** how do geometric and physicochemical environment fields differ between two protein conformations?

- **1CFD:** apo / calcium-free source structure.
- **1CLL:** holo / calcium-bound source structure; the bundled selection contains only protein heavy atoms, so we call the calculation **holo-derived**. Calcium, water, hydrogen and other heteroatoms are omitted.

Use active `mapsy.biomapsy` (`biomapsy.py`), with no periodic wrapping. Start with radial structural/compositional AC, radius 10 Å, orders 0–11 (24 coefficients), and add your notebook’s two physicochemical channels. Later, compare radial vs radial+angular on identical points.

**Charge field:** sum of the original lookup q/r within 8 Å, using a 0.1 Å distance floor. It is a truncated heavy-atom proxy, not a complete force-field potential. **Hydrophobicity:** atom-weighted inverse-distance average of residue Kyte–Doolittle values within 5 Å. Atom names/residues are aligned exactly with PDB coordinates; unknown-charge coverage is recorded.

Every state and boundary gets descriptors, PCA, both K curves and both selected maps. No fixed protein k=8 is imposed.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image
candidates = [Path.cwd() / 'examples', Path.cwd(), Path.cwd().parent]
EXAMPLES = next((p.resolve() for p in candidates if (p / 'common.py').is_file()), None)
if EXAMPLES is None:
    raise RuntimeError('Open this notebook from the repository root or examples folder.')
sys.path[:0] = [str(EXAMPLES), str(EXAMPLES / 'periodic_surfaces'), str(EXAMPLES / 'biomapsy'), str(EXAMPLES / 'cpu_gpu')]
from common import environment, show_analysis
print(environment())


In [ ]:
from protein_workflow import run
USE_GPU = False
summaries = {}
for protein, state in [('1CFD', 'apo'), ('1CLL', 'holo-derived')]:
    summary = run(protein, mode='tan', use_gpu=USE_GPU, ntries=2)
    summaries[(protein, 'tan')] = summary
    print(state, summary['physicochemical_fields'])
    show_analysis(summary)

## Compare states in the same PCA coordinate system

The per-system PCA above is useful for each map but its axes are fitted independently. The comparison below uses **one pooled scaler and one pooled PCA basis** for both states, with common color limits. It also shows the physicochemical distributions. Sampling points are spatially correlated; these distributions are descriptive, not independent biological replicates.

In [ ]:
from state_comparison import compare_states
comparison = compare_states(summaries[('1CFD','tan')], summaries[('1CLL','tan')])
folder = Path(comparison['output'])
for name in ['apo_shared_pca','holo_shared_pca','shared_pca_scatter','physicochemical_distributions']:
    display(Image(filename=str(folder/(name+'.png'))))
import pandas as pd
display(pd.read_csv(folder/'physicochemical_statistics.csv'))

## Check the second aperiodic boundary

`ionic-ass` was developed as the aperiodic version of ionic. It defines a different surface from `tan`; it need not produce the same positions. Retain the largest connected region and repeat exactly the same analysis. All before/after region counts and boundary parameters are saved.

In [ ]:
for protein in ['1CFD','1CLL']:
    summary = run(protein, mode='ionic-ass', use_gpu=USE_GPU, ntries=2)
    summaries[(protein, 'ionic-ass')] = summary
    show_analysis(summary)

## What happens when angular or physicochemical information is added?

Use identical 512 tan-surface positions per conformation to compare radial, radial+physicochemical, radial+angular, and all channels. Keep k fixed to that state’s full-data radial+physicochemical DB choice to isolate descriptor changes. These are subset controls, not new full-surface fits. Scores in different standardized feature spaces are not a ranking of physical accuracy.

In [ ]:
from state_comparison import assess_descriptor_sets
for protein in ['1CFD','1CLL']:
    assessment = assess_descriptor_sets(summaries[(protein,'tan')], points=512)
    print(protein, assessment['output'])
    display(pd.DataFrame(assessment['cases']))

## Interpretation

Protein PCA values are not constant. Opposite 3D views avoid superposing all front/back points, and the color bars identify 2–98% display limits. Full score ranges and loadings remain saved. Fields and cluster partitions describe the selected protein-only geometries; they do not isolate calcium chemistry or prove an allosteric mechanism.